# Mini Project: E-books and Reference Books

The Library application, stage 10. [Read this section](/books/001-python-programming-from-first-principles/inheritance.html).

Continue your `library.py` from [stage 09](/books/001-python-programming-from-first-principles/composition-has-a-relationship.html#library-project-09):

1. Add the subclasses `EBook` and `ReferenceBook` from this section.
2. Check that `Library.lend()` calculates the due date with `book.loan_days`, so that each kind of book gets its own loan period without changes to `Library`.
3. Add the e-book *B003 City Letters* by K. Owusu (2019, EPUB) and the reference book *B004 Atlas of Ghana* by Survey Office (2015) to the catalogue.
4. Lend B001 and B003 to Ada on 2 March 2026 and print each title with its due date. Try to lend B004 to Kofi and print the refusal.
5. Print whether the e-book is an `EBook` and a `Book`, and print its description.

The program should display:

```text
River Stories due 2026-03-16
City Letters due 2026-03-23
Refused: B004 is for reference only
True True
City Letters by K. Owusu (2019) [EPUB]
```

Compare your program with the [stage 10 checkpoint](/books/001-python-programming-from-first-principles/assets/inheritance/library_stage_10.py). `Library` did not change at all: it asks each book for its loan period and lets each book apply its own checkout rule.

Your `library.py` starts as the stage 09 checkpoint. If you have your own version, paste it below the `%%writefile library.py` line instead. Change the code step by step; running the cell saves the file.

In [ ]:
%%writefile library.py
"""Library stage 09: a Library has a catalogue, members and loans."""
from datetime import date, timedelta

class Book:
    loan_days = 14
    created_count = 0

    def __init__(self, copy_id, title, author, year):
        self.copy_id = copy_id
        self.title = title
        self.author = author
        self.year = year
        self._available = True
        Book.created_count += 1

    @classmethod
    def from_record(cls, record):
        copy_id, title, author, year = record.split(";")
        return cls(copy_id, title, author, int(year))

    @staticmethod
    def is_valid_copy_id(copy_id):
        return len(copy_id) == 4 and copy_id[0] == "B" and copy_id[1:].isdigit()

    @property
    def available(self):
        return self._available

    def description(self):
        return f"{self.title} by {self.author} ({self.year})"

    def check_out(self):
        if not self._available:
            raise ValueError(f"{self.copy_id} is already on loan")
        self._available = False

    def check_in(self):
        if self._available:
            raise ValueError(f"{self.copy_id} is not on loan")
        self._available = True


class Catalogue:
    def __init__(self):
        self._books = {}

    def add(self, book):
        if not Book.is_valid_copy_id(book.copy_id):
            raise ValueError(f"Invalid copy ID: {book.copy_id}")
        if book.copy_id in self._books:
            raise ValueError(f"Duplicate copy ID: {book.copy_id}")
        self._books[book.copy_id] = book

    def find(self, copy_id):
        return self._books.get(copy_id)

    def by_author(self, author):
        return [book.title for book in self._books.values() if book.author == author]

    def available_titles(self):
        return [book.title for book in self._books.values() if book.available]


class Member:
    def __init__(self, member_id, name):
        self.member_id = member_id
        self.name = name


class Loan:
    def __init__(self, book, member, start, due):
        self.book = book
        self.member = member
        self.start = start
        self.due = due
        self.returned = None

    @property
    def active(self):
        return self.returned is None

    def days_late(self, on_date):
        return max(0, (on_date - self.due).days)


class Library:
    def __init__(self, name):
        self.name = name
        self.catalogue = Catalogue()
        self._members = {}
        self._loans = []

    def add_member(self, member):
        if member.member_id in self._members:
            raise ValueError(f"Duplicate member ID: {member.member_id}")
        self._members[member.member_id] = member

    def lend(self, copy_id, member_id, today):
        book = self.catalogue.find(copy_id)
        member = self._members.get(member_id)
        if book is None or member is None:
            raise ValueError("Unknown book or member")
        book.check_out()
        loan = Loan(book, member, today, today + timedelta(days=book.loan_days))
        self._loans.append(loan)
        return loan

    def return_book(self, copy_id, today):
        for loan in self._loans:
            if loan.active and loan.book.copy_id == copy_id:
                loan.book.check_in()
                loan.returned = today
                return loan
        raise ValueError(f"{copy_id} has no active loan")

    def active_titles(self, member_id):
        return [loan.book.title for loan in self._loans
                if loan.active and loan.member.member_id == member_id]


def build_library():
    library = Library("Central")
    for record in [
        "B001;River Stories;A. Mensah;2024",
        "B002;Silent Maps;L. Chen;2021",
    ]:
        library.catalogue.add(Book.from_record(record))
    library.add_member(Member("M001", "Ada"))
    library.add_member(Member("M002", "Kofi"))
    return library


if __name__ == "__main__":
    library = build_library()
    loan = library.lend("B001", "M001", date(2026, 3, 2))
    print(loan.member.name, "borrowed", loan.book.title, "until", loan.due)
    print(library.active_titles("M001"))

    returned = library.return_book("B001", date(2026, 3, 20))
    print("Days late:", returned.days_late(date(2026, 3, 20)))
    print(library.catalogue.available_titles())


Run your program, as `python library.py` does in a terminal:

In [ ]:
%run library.py


Compare its output with the target output in the task. When you have finished, compare your program with the [stage 10 checkpoint](/books/001-python-programming-from-first-principles/assets/inheritance/library_stage_10.py).